<a href="https://colab.research.google.com/github/p-tech/wbs-dm-2026/blob/main/web_scrape/Scrape_Elements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## STEP 1: Inspect Web Page:
Open up the webpage in Chrome:  https://books.toscrape.com



# STEP 2: Inspect the page
1. Right click on a quote
2. Click Inspect
3. Indentify elements






# STEP 3: Install Required Library
We need to install for follwoing libraries.


*   request - enable us to call a web url
*   beautifulsoup4 - enables us to read the HTML




In [1]:
pip install requests beautifulsoup4

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

Which of these libraries:

Handles structure?

Handles downloading?

# STEP 5: Inspect the website
Download the page



In [13]:
url = "http://books.toscrape.com"
response = requests.get(url)
response.encoding = "utf-8"

print(response.status_code)

200


Response codes:

*   200 - OK - returned, loaded
*   301, 302 - redirected page
*   404 - error, page not found
*   500 - server error



# Parse the HTML

Take note of the structure






In [4]:
soup = BeautifulSoup(response.text, "lxml")

print(soup.title)
print(soup.title.text)

<title>
    All products | Books to Scrape - Sandbox
</title>

    All products | Books to Scrape - Sandbox



# Extract structured data

*   Book title
*   Price
*   Rating
*   Availability



# Find All Book Containers

In [5]:
books = soup.find_all("article", class_="product_pod")

print(len(books))

20


# Extract One Book Example

In [6]:
book = books[0]

title = book.h3.a["title"]
price = book.find("p", class_="price_color").text
availability = book.find("p", class_="instock availability").text.strip()
rating = book.p["class"][1]

print(title, price, availability, rating)


A Light in the Attic £51.77 In stock Three


Questions:

*   Why is the price text instead of a numeric?
*   Would this cuase a problem when storing in a database?
*   Why is price shown as Â£51.77





Need to add encoding on the response:

response.encoding = "utf-8"

# Extract all the books

In [7]:
data = []

for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    availability = book.find("p", class_="instock availability").text.strip()
    rating = book.p["class"][1]

    data.append([title, price, availability, rating])

df = pd.DataFrame(data, columns=["Title", "Price", "Availability", "Rating"])

df.head()

,Title,Price,Availability,Rating
0,A Light in the Attic,£51.77,In stock,Three
1,Tipping the Velvet,£53.74,In stock,One
2,Soumission,£50.10,In stock,One
3,Sharp Objects,£47.82,In stock,Four
4,Sapiens: A Brief History of Humankind,£54.23,In stock,Five


# Data Cleaning
Clean the price field.

Needs to be numeric so we can use for calculations.

In [8]:
df["Price"] = df["Price"].str.replace("£", "")
df["Price"] = pd.to_numeric(df["Price"])

Clean availability

Change to a boolean:

WHERE availability LIKE '%In stock%' - not as clean as WHERE InStock = 1

In [9]:
df["InStock"] = df["Availability"].str.contains("In stock")
df.drop(columns=["Availability"], inplace=True)

df.head()

,Title,Price,Rating,InStock
0,A Light in the Attic,51.77,Three,True
1,Tipping the Velvet,53.74,One,True
2,Soumission,50.10,One,True
3,Sharp Objects,47.82,Four,True
4,Sapiens: A Brief History of Humankind,54.23,Five,True


# Convert Rating to Numeric

In [10]:
df.dtypes

,0
Title,object
Price,float64
Rating,object
InStock,bool


In [11]:
df["Rating"] = df["Rating"].astype(object)
df.dtypes
print(df["Rating"].head())

0    Three
1      One
2      One
3     Four
4     Five
Name: Rating, dtype: object


In [12]:
# Reconstruct DataFrame from original data
#df = pd.DataFrame(data, columns=["Title", "Price", "Availability", "Rating"])

# Define the rating map
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# Convert 'Rating' to numeric using the map
df["Rating"] = df["Rating"].str.strip().map(rating_map)

# Display the head of the cleaned DataFrame
df.head()

,Title,Price,Rating,InStock
0,A Light in the Attic,51.77,3,True
1,Tipping the Velvet,53.74,1,True
2,Soumission,50.10,1,True
3,Sharp Objects,47.82,4,True
4,Sapiens: A Brief History of Humankind,54.23,5,True


# Prepare for a database
Add a primary key

In [ ]:
df.insert(0, "BookID", range(1, len(df) + 1))

df.head()

,BookID,Title,Price,Rating,InStock
0,1,A Light in the Attic,51.77,3,True
1,2,Tipping the Velvet,53.74,1,True
2,3,Soumission,50.10,1,True
3,4,Sharp Objects,47.82,4,True
4,5,Sapiens: A Brief History of Humankind,54.23,5,True


# Export to CSV

In [ ]:
df.to_csv("books_cleaned.csv", index=False)